# Kaggle Maze Crawler Starter Agent

A compact, notebook-first baseline for the Maze Crawler `crawl` environment. Run the notebook on Kaggle to generate `main.py` and `submission.py` from the same agent source, then verify both files before submitting.


## 1. Crawl Getting Started

This notebook is the source of truth for the Kaggle Maze Crawler starter agent. The baseline stays intentionally small:

1. `agent_v1` moves only the factory, prioritizing northward survival.
2. `agent_v2` adds one scout that collects visible crystals and returns energy to the factory.


## 2. Optional Visual Context

Pilkwang's structure-baseline notebook includes useful diagrams for thinking about factory survival, transfers, jumps, and opening choices. If the public figure dataset is attached to the Kaggle notebook, the next cell displays those images. If not, the notebook simply continues.


In [ ]:
# OPTIONAL_FIGURES
from pathlib import Path
from IPython.display import Image, display

figure_roots = [
    Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures"),
    Path("/kaggle/input/pilkwang-public-dataset-for-notebooks-figures"),
    Path("Figures"),
]
figure_names = ["Maze_Main.png", "Maze_Fig3.png", "Maze_Fig5.png", "Maze_Fig6.png", "Maze_Fig8.png", "Maze_Fig9.png"]

shown = 0
for name in figure_names:
    for root in figure_roots:
        path = root / name
        if path.exists():
            display(Image(filename=str(path)))
            shown += 1
            break
print(f"Displayed {shown} optional figure(s).")


## 3. Agent Helpers

Maze walls are stored as bitfields in a flat array. The helpers below keep the coordinate math, wall checks, and robot filtering in one place. They also work with either Kaggle's object-style observations or plain dictionaries, which makes local checks easier.


In [ ]:
DIRS = {"NORTH": (0, 1), "SOUTH": (0, -1), "EAST": (1, 0), "WEST": (-1, 0)}
WALL_BIT = {"NORTH": 1, "EAST": 2, "SOUTH": 4, "WEST": 8}


def value(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def wall_at(obs, config, col, row):
    """Wall bitfield at (col, row). Returns 0 if the cell is undiscovered."""
    walls = value(obs, "walls", [])
    south_bound = value(obs, "southBound", 0)
    width = value(config, "width", 0)
    idx = (row - south_bound) * width + col
    if 0 <= idx < len(walls) and walls[idx] != -1:
        return walls[idx]
    return 0


def has_road(obs, config, col, row, direction):
    """True if there is no wall in `direction` from (col, row)."""
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def neighbor(col, row, direction):
    """Return the (col, row) one step in `direction`."""
    dc, dr = DIRS[direction]
    return col + dc, row + dr


def my_robots(obs):
    """Map of uid -> robot data for robots owned by us."""
    player = value(obs, "player", 0)
    robots = value(obs, "robots", {}) or {}
    return {uid: data for uid, data in robots.items() if data[4] == player}


def my_factory(obs):
    """Return (uid, data) of our factory, or (None, None)."""
    for uid, data in my_robots(obs).items():
        if data[0] == 0:
            return uid, data
    return None, None


def parse_cell(key):
    """Parse a 'col,row' key into (col, row)."""
    col, row = key.split(",")
    return int(col), int(row)


## 4. Factory Bug Move

The factory's first job is to stay ahead of the scrolling boundary. This baseline moves north whenever possible, jumps over a north wall when the jump cooldown allows it, and otherwise tries a horizontal sidestep.


In [ ]:
def factory_bug_north(uid, col, row, jump_cd, obs, config):
    """Factory policy: move north, jump a north wall if ready, otherwise sidestep."""
    if has_road(obs, config, col, row, "NORTH"):
        return "NORTH"
    if jump_cd == 0:
        return "JUMP_NORTH"
    if has_road(obs, config, col, row, "EAST"):
        return "EAST"
    if has_road(obs, config, col, row, "WEST"):
        return "WEST"
    return "IDLE"


def agent_v1(obs, config):
    """Factory-only baseline for a quick survival smoke test."""
    actions = {}
    for uid, data in my_robots(obs).items():
        rtype, col, row = data[0], data[1], data[2]
        jump_cd = data[6] if len(data) > 6 else 0
        if rtype == 0:
            actions[uid] = factory_bug_north(uid, col, row, jump_cd, obs, config)
    return actions


## 5. Agent V1 Smoke Test

This cell runs only when the `crawl` environment is available. It is useful on Kaggle and harmless locally because unavailable environments are reported as skipped.


In [ ]:
try:
    from kaggle_environments import make

    env = make("crawl", configuration={"randomSeed": 42}, debug=True)
    env.run([agent_v1, "random"])
    for i, step in enumerate(env.steps[-1]):
        print(f"Player {i}: reward={step.reward}, status={step.status}")
    env.render(mode="ipython", width=800, height=800)
except Exception as exc:
    print("Smoke test skipped:", repr(exc))


## 6. Scout Snail Move

The scout uses a tiny memory: it avoids stepping onto either of the last two cells it visited. That is enough to reduce simple back-and-forth loops while keeping the policy easy to inspect.


In [ ]:
_scout_state = {}


def snail_step(uid, col, row, target, obs, config):
    """Greedy step toward `target`, refusing the last two cells visited by this scout."""
    state = _scout_state.setdefault(uid, {"target": None, "tabu": []})
    if state["target"] != target:
        state["target"] = target
        state["tabu"] = []

    tc, tr = target
    candidates = []
    for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
        if not has_road(obs, config, col, row, direction):
            continue
        nc, nr = neighbor(col, row, direction)
        if (nc, nr) in state["tabu"]:
            continue
        dist = abs(nc - tc) + abs(nr - tr)
        candidates.append((dist, direction))

    if not candidates:
        state["tabu"] = []
        return "IDLE"

    candidates.sort()
    action = candidates[0][1]
    state["tabu"] = (state["tabu"] + [(col, row)])[-2:]
    return action


def scout_action(uid, col, row, energy, factory_pos, obs, config):
    fc, fr = factory_pos
    if energy >= 75:
        for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
            if neighbor(col, row, direction) == (fc, fr) and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"
        return snail_step(uid, col, row, (fc, fr), obs, config)

    crystals = [parse_cell(key) for key in (value(obs, "crystals", {}) or {})]
    if crystals:
        target = min(crystals, key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row))
    else:
        target = (col, row + 5)
    return snail_step(uid, col, row, target, obs, config)


def agent_v2(obs, config):
    """Factory bug-move plus one scout for crystal collection and energy return."""
    actions = {}
    robots = my_robots(obs)
    _, f_data = my_factory(obs)
    scout_count = sum(1 for data in robots.values() if data[0] == 1)

    for uid, data in robots.items():
        rtype, col, row, energy = data[0], data[1], data[2], data[3]
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0

        if rtype == 0:
            if scout_count == 0 and build_cd == 0 and energy >= value(config, "scoutCost", 50):
                actions[uid] = "BUILD_SCOUT"
            else:
                actions[uid] = factory_bug_north(uid, col, row, jump_cd, obs, config)
        elif rtype == 1 and f_data is not None:
            factory_pos = (f_data[1], f_data[2])
            actions[uid] = scout_action(uid, col, row, energy, factory_pos, obs, config)

    return actions


## 7. Agent V2 Smoke Test

This smoke test runs the factory-plus-scout policy against a random opponent when Kaggle's `crawl` environment is available.


In [ ]:
try:
    from kaggle_environments import make

    _scout_state.clear()
    env = make("crawl", configuration={"randomSeed": 42}, debug=True)
    env.run([agent_v2, "random"])
    for i, step in enumerate(env.steps[-1]):
        print(f"Player {i}: reward={step.reward}, status={step.status}")
    env.render(mode="ipython", width=800, height=800)
except Exception as exc:
    print("Smoke test skipped:", repr(exc))


## 8. Generate Submission Files

Kaggle imports `agent` from `main.py`. This cell writes the final agent source to `main.py`; the next cell mirrors that exact source to `submission.py` for tooling that expects the alternate filename.


In [ ]:
%%writefile main.py
DIRS = {"NORTH": (0, 1), "SOUTH": (0, -1), "EAST": (1, 0), "WEST": (-1, 0)}
WALL_BIT = {"NORTH": 1, "EAST": 2, "SOUTH": 4, "WEST": 8}


def value(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def wall_at(obs, config, col, row):
    walls = value(obs, "walls", [])
    south_bound = value(obs, "southBound", 0)
    width = value(config, "width", 0)
    idx = (row - south_bound) * width + col
    if 0 <= idx < len(walls) and walls[idx] != -1:
        return walls[idx]
    return 0


def has_road(obs, config, col, row, direction):
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def neighbor(col, row, direction):
    dc, dr = DIRS[direction]
    return col + dc, row + dr


def my_robots(obs):
    player = value(obs, "player", 0)
    robots = value(obs, "robots", {}) or {}
    return {uid: data for uid, data in robots.items() if data[4] == player}


def my_factory(obs):
    for uid, data in my_robots(obs).items():
        if data[0] == 0:
            return uid, data
    return None, None


def parse_cell(key):
    c, r = key.split(",")
    return int(c), int(r)


def factory_bug_north(uid, col, row, jump_cd, obs, config):
    if has_road(obs, config, col, row, "NORTH"):
        return "NORTH"
    if jump_cd == 0:
        return "JUMP_NORTH"
    if has_road(obs, config, col, row, "EAST"):
        return "EAST"
    if has_road(obs, config, col, row, "WEST"):
        return "WEST"
    return "IDLE"


_scout_state = {}


def snail_step(uid, col, row, target, obs, config):
    state = _scout_state.setdefault(uid, {"target": None, "tabu": []})
    if state["target"] != target:
        state["target"] = target
        state["tabu"] = []

    tc, tr = target
    candidates = []
    for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
        if not has_road(obs, config, col, row, direction):
            continue
        nc, nr = neighbor(col, row, direction)
        if (nc, nr) in state["tabu"]:
            continue
        dist = abs(nc - tc) + abs(nr - tr)
        candidates.append((dist, direction))

    if not candidates:
        state["tabu"] = []
        return "IDLE"

    candidates.sort()
    action = candidates[0][1]
    state["tabu"] = (state["tabu"] + [(col, row)])[-2:]
    return action


def scout_action(uid, col, row, energy, factory_pos, obs, config):
    fc, fr = factory_pos
    if energy >= 75:
        for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
            if neighbor(col, row, direction) == (fc, fr) and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"
        return snail_step(uid, col, row, (fc, fr), obs, config)

    crystals = [parse_cell(key) for key in (value(obs, "crystals", {}) or {})]
    if crystals:
        target = min(crystals, key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row))
    else:
        target = (col, row + 5)
    return snail_step(uid, col, row, target, obs, config)


def compute_actions(obs, config):
    actions = {}
    robots = my_robots(obs)
    _, f_data = my_factory(obs)

    scout_count = sum(1 for data in robots.values() if data[0] == 1)

    for uid, data in robots.items():
        rtype, col, row, energy = data[0], data[1], data[2], data[3]
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0

        if rtype == 0:
            if scout_count == 0 and build_cd == 0 and energy >= value(config, "scoutCost", 50):
                actions[uid] = "BUILD_SCOUT"
            else:
                actions[uid] = factory_bug_north(uid, col, row, jump_cd, obs, config)
        elif rtype == 1 and f_data is not None:
            factory_pos = (f_data[1], f_data[2])
            actions[uid] = scout_action(uid, col, row, energy, factory_pos, obs, config)

    return actions


def idle_actions(obs):
    return {uid: "IDLE" for uid in my_robots(obs)}


def agent(obs, config):
    try:
        return compute_actions(obs, config)
    except Exception:
        return idle_actions(obs)


def act(obs, config):
    return agent(obs, config)


__all__ = ["agent", "act", "compute_actions"]


## 9. Verify Generated Files

The generated files should compile and stay byte-for-byte identical. Keep this check close to the write cell so stale submission artifacts are obvious.


In [ ]:
from pathlib import Path
import py_compile

main_path = Path("main.py")
submission_path = Path("submission.py")
submission_path.write_text(main_path.read_text(encoding="utf-8"), encoding="utf-8")

py_compile.compile(str(main_path), doraise=True)
py_compile.compile(str(submission_path), doraise=True)
assert main_path.read_text(encoding="utf-8") == submission_path.read_text(encoding="utf-8")

print("main.py: OK")
print("submission.py: OK")
print("submission.py sync: OK")


## 10. Submit To The Leaderboard

Run the generation and verification cells, save the Kaggle notebook, then use **Submit to Competition** from the Kaggle notebook panel. Kaggle will use the generated `main.py`; `submission.py` is available for downloads, local checks, and helper tooling.


## 11. Next Improvements

- Add a worker to remove walls in front of the factory with `REMOVE_NORTH`.
- Add miners and `TRANSFORM` them on mining nodes once you can recover the energy.
- Split multiple scouts across different crystals instead of sending everyone to the same target.
- Replace snail move with BFS or A* after caching enough wall information.
